In [33]:
import numpy as np

b = b"98"
np.array(b, dtype=np.dtype("U1"))
# struct.unpack('c',b)

array('9', dtype='<U1')

In [34]:
np.array(chr(int(b)))

array('b', dtype='<U1')

In [35]:
np.char.decode(b, "ascii")

array('98', dtype='<U2')

In [36]:
bytes = bytearray(b"\x01\x02\x03\x04\x05\x06\x05\x06")
st = np.dtype("i4, i2, i2")

In [37]:
arr = np.array(bytes, dtype=st)

In [38]:
arr["f0"]

array([1, 2, 3, 4, 5, 6, 5, 6])

In [39]:
bytes = bytearray(b"\x01\x02\x03\x05\x06\x05")
st = np.dtype("i2, i2, i2")

np.frombuffer(bytes, dtype=st)

array([(513, 1283, 1286)],
      dtype=[('f0', '<i2'), ('f1', '<i2'), ('f2', '<i2')])

In [40]:
st.itemsize, len(bytes)

(6, 6)

In [41]:
rows = 2
columns = 3
bytes = bytearray(b"\x01\x02\x03\x04 \x05\x06\x05\x06")
st = np.dtype("i2, i2, i2")
np.recarray((rows, columns), dtype=st, buf=bytes)

TypeError: buffer is too small for requested array

In [42]:
ss = "123 12341 123123123"
st = np.dtype("i4, i4, i4")
np.fromstring(ss, st, count=1, sep=" ")

ValueError: don't know how to read character strings with that array type

In [43]:
np.loadtxt([ss], dtype=st)

array((123, 12341, 123123123),
      dtype=[('f0', '<i4'), ('f1', '<i4'), ('f2', '<i4')])

In [44]:
np.loadtxt([ss], dtype=st, unpack=True)

[array(123), array(12341), array(123123123)]

In [49]:
a = np.genfromtxt([ss], dtype=st)
a["f0"]

array(123)

In [53]:
list(a.flat)

[(123, 12341, 123123123)]

In [22]:
import numpy as np
import pandas as pd
import io

test_str = [
    '"blasdasdasdas \\" asdasda \\" asdasd " 1231322 1.0e-10 a 1.2   !comment\n',
    '"blasdasdasdas" 12313 1.2e-10 a 1.6    !comment2',
]

In [24]:
df = pd.read_table(io.StringIO(test_str[0]), sep="\s+", comment="!", header=None, escapechar="\\", engine="c")

In [31]:
df

,0,1,2,3,4
0,"blasdasdasdas "" asdasda "" asdasd",1231322,1.000000e-10,a,1.2


In [47]:
test_bool_list = [True, False, True]
np.where(test_bool_list)[0]

array([0, 2], dtype=int64)

In [29]:
buf = io.TextIOWrapper(io.BytesIO(test_str[0].encode("ascii")), encoding="ascii")

In [30]:
buf.read(10)

'"blasdasda'

In [49]:
buf = np.array([90], dtype="i1")

In [54]:
buf.view("S1")

array([b'Z'], dtype=object)

In [56]:
buf11 = buf.view("S1").astype(object)
buf11[0]

b'Z'

In [61]:
buf12 = np.char.decode(buf.view("S1"), "ascii")
buf12

array(['Z'], dtype='<U1')

In [60]:
buf2 = np.array([90], dtype="c")
buf2

array([b'9'], dtype='|S1')

In [2]:
bstr = "a".encode("ascii")

In [3]:
bstr

b'a'

In [82]:
import numpy as np
import io

arr = np.array([1, 2, 123231, 6], dtype=np.dtype(">i4"))

In [83]:
arr[0]

1

In [84]:
arr.dtype, type(arr[0])

(dtype('>i4'), numpy.int32)

In [85]:
arr.data[0:1]

In [86]:
buf = io.BytesIO()
buf.write(arr[0])

4

In [87]:
buf.getvalue()

b'\x01\x00\x00\x00'

In [93]:
buf = io.BytesIO()
buf.write(arr.data[1:1])

0

In [94]:
buf.getvalue()

b''

In [95]:
# String array to char array
str_arr = np.array(["a", "b", "x", "y", "z"], dtype=object)
print(str_arr.dtype)

object


In [103]:
str_arr.astype("S1").view("i1")

array([ 97,  98, 120, 121, 122], dtype=int8)

In [108]:
repr(str_arr.astype("S1").view("i1"))

'array([ 97,  98, 120, 121, 122], dtype=int8)'

In [67]:
# 80 bit floating point value according to the IEEE-754 specification and the Standard Apple Numeric Environment specification:
# 1 bit sign, 15 bit exponent, 1 bit normalization indication, 63 bit mantissa
buffer = bytearray.fromhex("00000000000000800040000000000000")
buffer = bytearray.fromhex("000000000000E4C00C40000000000000")
buffer = buffer[:10]
buffer.reverse()
print(repr(buffer))
if (buffer[0] & 0x80) == 0x00:
    sign = 1
else:
    sign = -1
sign

bytearray(b'@\x0c\xc0\xe4\x00\x00\x00\x00\x00\x00')


1

In [68]:
int("80", 16)

128

In [69]:
exponent = ((buffer[0] & 0x7F) << 8) | buffer[1]
exponent

16396

In [70]:
mantissa = buffer[2:]
print(mantissa)
if (mantissa[0] & 0x80) != 0x00:
    normalizeCorrection = 1
else:
    normalizeCorrection = 0
normalizeCorrection

bytearray(b'\xc0\xe4\x00\x00\x00\x00\x00\x00')


1

In [73]:
m2 = int.from_bytes(mantissa, "big") & 0x7FFFFFFFFFFFFFFF
m2

4675862313117417472

In [74]:
# value = (-1) ^ s * (normalizeCorrection + m / 2 ^ 63) * 2 ^ (e - 16383)
import math

sign * (normalizeCorrection + float(m2 / (1 << 63))) * math.pow(2, exponent - 16383)

12345.0

In [64]:
float(m2 / (1 << 63))

1.3877787807814457e-17

In [65]:
math.pow(2, exponent - 16383)

2.0